# Gemma Lie Detector - 02 Tabular (German credit)
Same protocol as notebook 01, but the elements are **feature columns** (ablation = "not provided"). The extra question: which features really drive the decision (e.g. `age`, `sex`) versus which ones Gemma admits to using?
Dataset: Kaggle `uciml/german-credit` (upload the CSV to Colab), or the OpenML copy (no login).

In [ ]:
import os
if not os.path.exists("Gemma-Lie-Detector") and not os.path.exists("src"):
    !git clone https://github.com/colbyrobbins/Gemma-Lie-Detector.git
if os.path.exists("Gemma-Lie-Detector"):
    %cd Gemma-Lie-Detector
!pip -q install -U "transformers>=4.50" accelerate datasets scikit-learn tqdm scipy matplotlib
!pip -q install -e . --no-deps

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))   # needs the Gemma license accepted on this HF account

In [ ]:
N = 100
CSV_PATH = None     # e.g. "/content/german_credit_data.csv" after uploading the Kaggle file; None -> OpenML
MODEL_ID, DTYPE = "google/gemma-3-4b-it", "auto"

from gemma_lie_detector.model import GemmaScorer
from gemma_lie_detector.prompts import CREDIT
from gemma_lie_detector.data import load_credit_kaggle, load_credit_openml

scorer = GemmaScorer(CREDIT.labels, model_id=MODEL_ID, dtype=DTYPE)
examples = load_credit_kaggle(CSV_PATH, n=N) if CSV_PATH else load_credit_openml(n=N)
print(examples[0].render())

In [ ]:
ex = examples[0]
p = CREDIT.prompt(ex.render())
print(scorer.score([p]), scorer.generate([p])[0])   # smoke test: label mass ~1, valid JSON

In [ ]:
from gemma_lie_detector.experiments import run_core, run_cue
run_core(examples, CREDIT, scorer, "results/tab_core.jsonl")
run_cue(examples, CREDIT, scorer, "results/tab_cue.jsonl")

In [ ]:
from gemma_lie_detector.analysis import load_results, summarize_core, summarize_cue, feature_disclosure_table, make_figures
core, cue = load_results("results/tab_core.jsonl"), load_results("results/tab_cue.jsonl")
display(summarize_core(core).round(3)); display(summarize_cue(cue).round(3))
display(feature_disclosure_table(core).round(3))   # driver_rate vs cited_rate per feature
make_figures(core, cue, outdir="results/figures_tab")